# 03 - Advanced SQL
Topics: `HAVING`, CTEs (`WITH`), and window functions (`RANK`, `AVG() OVER`, `ROW_NUMBER`).

The database `hr_analytics.db` must exist. If it does not, run notebook 02 first.

## 1. Connect to the database

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/hr_analytics.db")

def run(sql):
    return pd.read_sql_query(sql, conn)

print("Connected")

Connected


## 2. Quick check that the data is there

In [2]:
run("SELECT COUNT(*) AS employees FROM employee")

,employees
0,1470


## 3. Queries
The same queries are saved in `sql/02_advanced_queries.sql`.

### 1. HAVING: filter groups, not rows
`WHERE` filters **rows before** grouping. `HAVING` filters **groups after** grouping. Here we keep only job roles that have at least 30 employees, because a rate calculated on 5 people is not reliable.

In [3]:
run("""
SELECT
    JobRole,
    COUNT(*) AS total_employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS employees_left,
    ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct
FROM employee
GROUP BY JobRole
HAVING COUNT(*) >= 30
ORDER BY attrition_rate_pct DESC;
""")

,JobRole,total_employees,employees_left,attrition_rate_pct
0,Sales Representative,83,33,39.8
1,Data Scientist,261,62,23.8
2,Sales Executive,327,57,17.4
3,Software Engineer,294,47,16.0
4,Senior Software Engineer,132,9,6.8
5,Machine Learning Engineer,146,10,6.8
6,Analytics Manager,52,3,5.8
7,Manager,37,2,5.4
8,Engineering Manager,75,2,2.7


### 2. WHERE and HAVING together
First `WHERE` keeps only employees who work overtime (row filter). Then we group by department, and `HAVING` keeps only departments where at least 10 of these employees left (group filter).

In [4]:
run("""
SELECT
    Department,
    COUNT(*) AS overtime_employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS employees_left
FROM employee
WHERE OverTime = 'Yes'
GROUP BY Department
HAVING SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) >= 10
ORDER BY employees_left DESC;
""")

,Department,overtime_employees,employees_left
0,Technology,271,74
1,Sales,128,48


### 3. CTE: compare each job role with the company average
We build two: `overall` (one row with the company rate) and `by_role` (one row per job role). Then we join them and calculate the difference. `CROSS JOIN` attaches the single `overall` row to every role.

In [5]:
run("""
WITH overall AS (
    SELECT 100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) AS overall_rate
    FROM employee
),
by_role AS (
    SELECT
        JobRole,
        COUNT(*) AS employees,
        100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*) AS role_rate
    FROM employee
    GROUP BY JobRole
    HAVING COUNT(*) >= 30
)
SELECT
    r.JobRole,
    r.employees,
    ROUND(r.role_rate, 1) AS role_rate_pct,
    ROUND(o.overall_rate, 1) AS company_rate_pct,
    ROUND(r.role_rate - o.overall_rate, 1) AS difference_pct_points
FROM by_role r
CROSS JOIN overall o
ORDER BY difference_pct_points DESC;
""")

,JobRole,employees,role_rate_pct,company_rate_pct,difference_pct_points
0,Sales Representative,83,39.8,16.1,23.6
1,Data Scientist,261,23.8,16.1,7.6
2,Sales Executive,327,17.4,16.1,1.3
3,Software Engineer,294,16.0,16.1,-0.1
4,Machine Learning Engineer,146,6.8,16.1,-9.3
5,Senior Software Engineer,132,6.8,16.1,-9.3
6,Analytics Manager,52,5.8,16.1,-10.4
7,Manager,37,5.4,16.1,-10.7
8,Engineering Manager,75,2.7,16.1,-13.5


### 4. Salary of people who left vs stayed, inside the same job role
This answers the question from my notes: is the salary gap only because of different jobs? `CASE WHEN` without `ELSE` gives `NULL` for the other group, and `AVG` ignores `NULL`, so each `AVG` only uses its own group.

In [6]:
run("""
SELECT
    JobRole,
    COUNT(*) AS employees,
    ROUND(AVG(CASE WHEN Attrition = 'Yes' THEN Salary END)) AS avg_salary_left,
    ROUND(AVG(CASE WHEN Attrition = 'No'  THEN Salary END)) AS avg_salary_stayed,
    ROUND(AVG(CASE WHEN Attrition = 'No'  THEN Salary END)
        - AVG(CASE WHEN Attrition = 'Yes' THEN Salary END)) AS gap
FROM employee
GROUP BY JobRole
HAVING COUNT(*) >= 30
ORDER BY gap DESC;
""")

,JobRole,employees,avg_salary_left,avg_salary_stayed,gap
0,Software Engineer,294,39756.0,54291.0,14534.0
1,Machine Learning Engineer,146,116822.0,131146.0,14324.0
2,Sales Representative,83,36443.0,43437.0,6994.0
3,Data Scientist,261,51095.0,57633.0,6538.0
4,Sales Executive,327,129358.0,114628.0,-14730.0
5,Analytics Manager,52,364018.0,345411.0,-18607.0
6,Senior Software Engineer,132,149830.0,124429.0,-25400.0
7,Manager,37,346082.0,315900.0,-30182.0
8,Engineering Manager,75,421679.0,282548.0,-139131.0


### 5. Window function: RANK inside each department

In [7]:
run("""
WITH role_salary AS (
    SELECT Department, JobRole, COUNT(*) AS employees, ROUND(AVG(Salary)) AS avg_salary
    FROM employee
    GROUP BY Department, JobRole
)
SELECT
    Department,
    JobRole,
    employees,
    avg_salary,
    RANK() OVER (PARTITION BY Department ORDER BY avg_salary DESC) AS salary_rank_in_department
FROM role_salary
ORDER BY Department, salary_rank_in_department;
""")

,Department,JobRole,employees,avg_salary,salary_rank_in_department
0,Human Resources,HR Manager,4,449331.0,1
1,Human Resources,HR Business Partner,7,314002.0,2
2,Human Resources,HR Executive,28,94362.0,3
3,Human Resources,Recruiter,24,37648.0,4
4,Sales,Manager,37,317531.0,1
5,Sales,Sales Executive,326,116575.0,2
6,Sales,Sales Representative,83,40656.0,3
7,Technology,Analytics Manager,52,346484.0,1
8,Technology,Sales Executive,1,319619.0,2
9,Technology,Engineering Manager,75,286259.0,3


### 6. Each employee vs the average of their job role
`AVG(Salary) OVER (PARTITION BY JobRole)` puts the average salary of the job role next to every employee, **keeping all rows**. Then we subtract it. The result shows the 10 employees paid the most above their role average.

In [8]:
run("""
SELECT
    EmployeeID,
    JobRole,
    Salary,
    ROUND(AVG(Salary) OVER (PARTITION BY JobRole)) AS role_avg_salary,
    ROUND(Salary - AVG(Salary) OVER (PARTITION BY JobRole)) AS difference_from_role_avg
FROM employee
ORDER BY difference_from_role_avg DESC
LIMIT 10;
""")

,EmployeeID,JobRole,Salary,role_avg_salary,difference_from_role_avg
0,372C-E204,Data Scientist,455643,56079.0,399564.0
1,801E-9D0E,Software Engineer,439641,51967.0,387674.0
2,AFC3-E23F,Data Scientist,423941,56079.0,367862.0
3,5E73-CE19,Software Engineer,314181,51967.0,262214.0
4,1A82-AF2C,Senior Software Engineer,380248,126161.0,254087.0
5,1A7C-19DB,Engineering Manager,539998,286259.0,253739.0
6,7B30-BAC8,Machine Learning Engineer,366078,130165.0,235913.0
7,DFE3-6FA7,Sales Executive,350525,117196.0,233329.0
8,8892-77D8,Engineering Manager,519361,286259.0,233102.0
9,1424-0B0C,Senior Software Engineer,356378,126161.0,230217.0


### 7. ROW_NUMBER: take only the latest review of each employee
Each employee can have several reviews. `ROW_NUMBER()` numbers the reviews of each employee from the newest (1) to the oldest. Keeping `review_number = 1` gives one row per employee. Then we compare the average manager rating of employees who left and who stayed. Only employees with at least one review are included (because of the plain `JOIN`).

In [10]:
run("""
WITH ranked_reviews AS (
    SELECT
        EmployeeID,
        ManagerRating,
        ROW_NUMBER() OVER (PARTITION BY EmployeeID ORDER BY ReviewDate DESC) AS review_number
    FROM performance
)
SELECT
    e.Attrition,
    COUNT(*) AS employees,
    ROUND(AVG(r.ManagerRating), 2) AS avg_latest_manager_rating
FROM ranked_reviews r
JOIN employee e ON r.EmployeeID = e.EmployeeID
WHERE r.review_number = 1
GROUP BY e.Attrition;
""")

,Attrition,employees,avg_latest_manager_rating
0,No,1043,3.46
1,Yes,237,3.41


## 4. Close the connection

In [11]:
conn.close()

## My notes

**What I found**
- Sales Representative has the highest attrition rate: 39.8% (33 of 83 employees), about 24 percentage points above the company rate of 16.1%. Data Scientist is second with 23.8% (62 of 261).
- The lowest rates are in management roles: Engineering Manager 2.7%, Manager 5.4%, and Analytics Manager 5.8%.
- Among employees who work overtime, 74 of 271 left in Technology and 48 of 128 left in Sales. Human Resources does not appear because fewer than 10 of its overtime employees left (the HAVING filter).
- Inside the same job role, the salary gap between employees who left and who stayed is not the same in every role. In 4 roles (Software Engineer, Machine Learning Engineer, Sales Representative, Data Scientist) employees who left earned less on average. In 5 roles (Sales Executive, Analytics Manager, Senior Software Engineer, Manager, Engineering Manager) employees who left earned more.
- So the overall gap I saw earlier (82,262 for leavers vs 118,856 for stayers) is partly explained by the mix of job roles: more employees left from roles with lower salaries, such as Sales Representative and Software Engineer.
- The average manager rating of the latest review is 3.46 for employees who stayed (1,043) and 3.41 for employees who left (237). This is a small difference.

**What I need to be careful about**
- These are relationships, not causes. I cannot say that low salary or overtime makes employees leave.
- I only used job roles with at least 30 employees (HAVING). Rates for small roles would be unreliable, but this also means some roles are not shown.
- Some salaries look extreme. For example, one Data Scientist earns 455,643 while the average for the role is 56,079. I must check the salary distribution for outliers or data errors before trusting averages.
- Only one Sales Executive is listed in the Technology department, which looks odd and should be checked.
- The latest-review comparison only includes employees who have at least one review (1,280 of 1,470).

**What I want to explore next**
- Check the salary distribution and outliers with Python (box plots and histograms).
- Use the median salary, not only the average, when outliers exist.
- Check attrition by age group and years since last promotion.
- Check whether overtime still matters inside each job role.
- Build charts for the main findings, then a Power BI dashboard.